In [6]:
import pandas as pd
import numpy as np

# ── Load data ──────────────────────────────────────────────────────────────
iati = pd.read_csv('/Users/jackzipper/QSS20/final_project/final_project_data/iati-drc-cleaned.csv')
conflict = pd.read_csv('/Users/jackzipper/QSS20/final_project/final_project_data/conflict_dat_cleaned.csv')

# ── Parse dates ────────────────────────────────────────────────────────────
iati['day_start'] = pd.to_datetime(iati['day_start'])
iati['day_end']   = pd.to_datetime(iati['day_end'])

conflict['date_start'] = pd.to_datetime(conflict['date_start'])
conflict['year_month']  = conflict['date_start'].dt.to_period('M')

# ── Filter to 2021–2026 ────────────────────────────────────────────────────
conflict = conflict[conflict['date_start'].dt.year.between(2021, 2026)]

cutoff_start = pd.Timestamp('2021-01-01')
cutoff_end   = pd.Timestamp('2026-12-31')
iati = iati[
    (iati['day_start'] <= cutoff_end) &
    (iati['day_end']   >= cutoff_start)
]

# ══════════════════════════════════════════════════════════════════════════
# PART 1 — Build the full balanced grid (every admin2 x every month)
# ══════════════════════════════════════════════════════════════════════════

# All admin2 units across both datasets
all_admin2 = pd.Index(
    pd.concat([
        conflict['town_admin2'].rename('admin2_name'),
        iati['admin2_name']
    ]).dropna().unique(),
    name='admin2_name'
)

# Every month in the window
all_months = pd.period_range(start='2021-01', end='2026-12', freq='M')

# Full cartesian product
grid = pd.MultiIndex.from_product(
    [all_admin2, all_months],
    names=['admin2_name', 'year_month']
).to_frame(index=False)

print(f"Balanced grid size: {len(grid):,} rows "
      f"({all_admin2.nunique()} admin2 units × {len(all_months)} months)")

# ══════════════════════════════════════════════════════════════════════════
# PART 2 — Conflict aggregation (admin2 x month)
# ══════════════════════════════════════════════════════════════════════════

conflict_monthly = (
    conflict
    .groupby(['town_admin2', 'year_month'])
    .agg(
        violent_incidents = ('id',   'count'),
        total_deaths      = ('best', 'sum')
    )
    .reset_index()
    .rename(columns={'town_admin2': 'admin2_name'})
)

# ══════════════════════════════════════════════════════════════════════════
# PART 3 — Aid aggregation (admin2 x month), fractional spend approach
# ══════════════════════════════════════════════════════════════════════════

iati_valid = iati.dropna(subset=['day_start', 'day_end', 'spend', 'admin2_name']).copy()
iati_valid['total_days'] = (iati_valid['day_end'] - iati_valid['day_start']).dt.days.clip(lower=1)

rows = []

for _, row in iati_valid.iterrows():
    effective_start = max(row['day_start'], cutoff_start)
    effective_end   = min(row['day_end'],   cutoff_end)

    months = pd.period_range(start=effective_start, end=effective_end, freq='M')

    for month in months:
        month_start = month.start_time
        month_end   = month.end_time

        overlap_start = max(row['day_start'], month_start)
        overlap_end   = min(row['day_end'],   month_end)
        active_days   = (overlap_end - overlap_start).days + 1

        fractional_spend = (active_days / row['total_days']) * row['spend']

        rows.append({
            'admin2_name'      : row['admin2_name'],
            'year_month'       : month,
            'aid'              : row['aid'],
            'fractional_spend' : fractional_spend
        })

aid_expanded = pd.DataFrame(rows)

aid_monthly = (
    aid_expanded
    .groupby(['admin2_name', 'year_month'])
    .agg(
        num_aid_projects = ('aid',              'nunique'),
        total_aid_spend  = ('fractional_spend', 'sum')
    )
    .reset_index()
)

# ══════════════════════════════════════════════════════════════════════════
# PART 4 — Merge everything onto the balanced grid
# ══════════════════════════════════════════════════════════════════════════

panel = (
    grid
    .merge(conflict_monthly, on=['admin2_name', 'year_month'], how='left')
    .merge(aid_monthly,      on=['admin2_name', 'year_month'], how='left')
)

# True zeros for months with no observations
panel['violent_incidents'] = panel['violent_incidents'].fillna(0).astype(int)
panel['total_deaths']      = panel['total_deaths'].fillna(0).astype(int)
panel['num_aid_projects']  = panel['num_aid_projects'].fillna(0).astype(int)
panel['total_aid_spend']   = panel['total_aid_spend'].fillna(0)

panel = panel.sort_values(['admin2_name', 'year_month']).reset_index(drop=True)

print(f"Panel shape         : {panel.shape}")
print(f"Unique admin2 units : {panel['admin2_name'].nunique()}")
print(f"Unique months       : {panel['year_month'].nunique()}")
print(f"Date range          : {panel['year_month'].min()} – {panel['year_month'].max()}")
print(f"Rows with conflict  : {(panel['violent_incidents'] > 0).sum():,}")
print(f"Rows with aid       : {(panel['num_aid_projects']  > 0).sum():,}")
print()
print(panel.head(10))

# ── Save ───────────────────────────────────────────────────────────────────
panel.to_csv('/Users/jackzipper/QSS20/final_project/final_project_data/violence_aid_merged.csv', index=False)
print("\nSaved to panel_admin2_monthly.csv")

Balanced grid size: 10,512 rows (146 admin2 units × 72 months)
Panel shape         : (10512, 6)
Unique admin2 units : 146
Unique months       : 72
Date range          : 2021-01 – 2026-12
Rows with conflict  : 684
Rows with aid       : 8,302

  admin2_name year_month  violent_incidents  total_deaths  num_aid_projects  \
0       Aketi    2021-01                  0             0                 0   
1       Aketi    2021-02                  0             0                 0   
2       Aketi    2021-03                  0             0                 0   
3       Aketi    2021-04                  0             0                 0   
4       Aketi    2021-05                  0             0                 0   
5       Aketi    2021-06                  0             0                 0   
6       Aketi    2021-07                  0             0                 0   
7       Aketi    2021-08                  0             0                 0   
8       Aketi    2021-09                  0    